<a href="https://colab.research.google.com/github/Andrea31-21/Curso2026-2027/blob/master/Assignment4/course_materials/notebooks/Task07_mandatory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 07: Querying RDF(s)**

In [1]:
!pip install rdflib
!pip install oeg-sw-class
github_storage = "https://raw.githubusercontent.com/FacultadInformatica-LinkedData/Curso2026-2027/master/Assignment4/course_materials"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 13.8 MB/s eta 0:00:00


Spanish: Primero leemos los ficheros RDF

English: First let's read the RDF file

In [2]:
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS
from oeg_sw_class import Report
# Do not change the name of the variables
g = Graph()
g.namespace_manager.bind('ns', Namespace("http://somewhere#"), override=False)
g.parse(github_storage+"/rdf/data07.ttl", format="TTL")
report = Report()

**TASK 7.1a:**

Spanish: Para todas las clases, enumera cada classURI. Si la clase pertenece a otra clase, indica su superclase. Realiza el ejercicio en RDFLib devolviendo una lista de tuplas: (clase, superclase) denominada "result". Si una clase no tiene superclase, devuelve None como superclase.

English: For all classes, list each classURI. If the class belogs to another class, then list its superclass. Do the exercise in RDFLib returning a list of Tuples: (class, superclass) called "result". If a class does not have a super class, then return None as the superclass

In [5]:
result = [] #list of tuples
for c in g.subjects(RDF.type, RDFS.Class):
  superclass = g.value(c, RDFS.subClassOf)
  result.append((c, superclass))

# Visualize the results
for r in result:
  print(r)

(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'), None)
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Animal'), None)
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Student'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#FullProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#AssistantProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#AssociateProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#InterimAssociateProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#AssociateProfessor'))


In [6]:
## Validation: Do not remove
report.validate_07_1a(result)

TASK 7.1a OK


**TASK 7.1b:**

Spanish: Repite el mismo ejercicio en SPARQL, devolviendo las variables ?c (clase) y ?sc (superclase)

English: Repeat the same exercise in SPARQL, returning the variables ?c (class) and ?sc (superclass)

In [11]:
query = """ SELECT ?c ?sc WHERE {
?c rdf:type rdfs:Class.
OPTIONAL { ?c rdfs:subClassOf ?sc }
}
"""

for r in g.query(query):
  print(r.c, r.sc)


http://oeg.fi.upm.es/def/people#Person None
http://oeg.fi.upm.es/def/people#Animal None
http://oeg.fi.upm.es/def/people#Professor http://oeg.fi.upm.es/def/people#Person
http://oeg.fi.upm.es/def/people#Student http://oeg.fi.upm.es/def/people#Person
http://oeg.fi.upm.es/def/people#FullProfessor http://oeg.fi.upm.es/def/people#Professor
http://oeg.fi.upm.es/def/people#AssistantProfessor http://oeg.fi.upm.es/def/people#Professor
http://oeg.fi.upm.es/def/people#AssociateProfessor http://oeg.fi.upm.es/def/people#Professor
http://oeg.fi.upm.es/def/people#InterimAssociateProfessor http://oeg.fi.upm.es/def/people#AssociateProfessor


In [12]:
## Validation: Do not remove
report.validate_07_1b(query,g)

TASK 7.1b OK


**TASK 7.2a:**

Spanish: Enumera todos los individuos de "Person" con RDFLib (ten en cuenta las subclases). Devuelve los URI de los individuos en una lista llamada "individuals".

English: List all individuals of "Person" with RDFLib (remember the subClasses). Return the individual URIs in a list called "individuals"


In [13]:
ns = Namespace("http://oeg.fi.upm.es/def/people#")

# variable to return
individuals = []

subclasses = {ns.Person}
for s, p, o in g.triples((None, RDFS.subClassOf, None)):
  for subclass in g.subjects(RDFS.subClassOf, ns.Person):
    subclasses.add(subclass)
    for sub_subclass in g.subjects(RDFS.subClassOf, subclass):
      subclasses.add(sub_subclass)

for c in subclasses:
  for ind in g.subjects(RDF.type, c):
    if ind not in individuals:
      individuals.append(ind)

# visualize results
for i in individuals:
  print(i)

http://oeg.fi.upm.es/def/people#Asun
http://oeg.fi.upm.es/def/people#Raul
http://oeg.fi.upm.es/def/people#Juan
http://oeg.fi.upm.es/def/people#Oscar


In [14]:
# validation. Do not remove
report.validate_07_02a(individuals)

TASK 7.2a OK


**TASK 7.2b:**

Spanish: Repite el mismo ejercicio en SPARQL, devolviendo los URI individuales en una variable ?ind.

English: Repeat the same exercise in SPARQL, returning the individual URIs in a variable ?ind

In [24]:
query = """
PREFIX ns: <http://oeg.fi.upm.es/def/people#>

SELECT DISTINCT ?ind WHERE {
?subclass rdfs:subClassOf* ns:Person .
?ind rdf:type ?subclass.
}
"""

for r in g.query(query):
  print(r.ind)
# Visualize the results

http://oeg.fi.upm.es/def/people#Asun
http://oeg.fi.upm.es/def/people#Raul
http://oeg.fi.upm.es/def/people#Juan
http://oeg.fi.upm.es/def/people#Oscar


In [25]:
## Validation: Do not remove
report.validate_07_02b(g, query)

TASK 7.2b OK


**TASK 7.3:**

Spanish: Enumera el nombre y el tipo de quienes conocen a Curry (solo en SPARQL). Utiliza el nombre y el tipo como variables en la consulta.

English: List the name and type of those who know Curry (in SPARQL only). Use name and type as variables in the query

In [42]:
query =  """
PREFIX ns: <http://oeg.fi.upm.es/def/people#>

SELECT DISTINCT ?name ?type WHERE {
?person ns:knows ns:Curry .
?person rdfs:label ?name .
?person rdf:type ?type .
}
"""

# Visualize the results
for r in g.query(query):
  print(r.name, r.type)


Juan http://oeg.fi.upm.es/def/people#AssistantProfessor


In [43]:
## Validation: Do not remove
report.validate_07_03(g, query)

TASK 7.3 OK


**Task 7.4:**

Spanish: Enumera los nombres de aquellas entidades que tengan un compañero de trabajo que tenga un perro, o que tengan un compañero de trabajo que tenga un compañero de trabajo que tenga un perro (en SPARQL). Devuelve los resultados en una variable llamada «name».

English: List the name of those entities who have a colleague with a dog, or that have a collegue who has a colleague who has a dog (in SPARQL). Return the results in a variable called name

In [52]:
query =  """
PREFIX ns: <http://oeg.fi.upm.es/def/people#>

SELECT DISTINCT ?name WHERE {
{
?person ns:hasColleague ?colleague .
?colleague ns:ownsPet ?dog .
}
UNION
{
?person ns:hasColleague ?colleague1 .
?colleague1 ns:hasColleague ?colleague2 .
?colleague2 ns:ownsPet ?dog .
}
?person rdfs:label ?name .
}
"""

for r in g.query(query):
  print(r.name)

# Visualize the results

Asun
Juan
Oscar
Raul


In [53]:
## Validation: Do not remove
report.validate_07_04(g,query)
report.save_report("_Task_07")

TASK 7.4 OK
